# Cvičné úlohy — řešení

Ke každé úloze: seznam chyb, opravený kód, rozšíření a výsledky ladění.
**Dívej se sem až po vlastním pokusu** — jinak si vyrobíš pocit znalosti bez znalosti.

---

## Úloha 1 — Odstranění záporných hodnot

*Archetyp: mazání během iterace (katalog #6)*

### Nalezené chyby

| # | Typ | Popis |
|---|-----|-------|
| 1 | sémantická | **Mazání během iterace.** `remove` posune zbytek seznamu doleva, iterátor si drží index → následující prvek se **přeskočí**. |
| 2 | sémantická | `remove` maže **první výskyt hodnoty**, ne prvek na aktuální pozici. U duplicit smaže něco jiného, než čekáš. |
| 3 | návrhová | Funkce modifikuje vstup volajícího a zároveň něco vrací — matoucí. |

**Co kód doopravdy udělá:** `[12,-1,-3,15,20,-7,18]` → smaže `-1`, seznam se posune, `-3` se
octne na už navštívené pozici a **přeskočí se**. Výsledek `[12,-3,15,20,18]` a vrácené `2` — obojí špatně.

### Opravené a rozšířené řešení

In [ ]:
def uklid_mereni(mereni, mez=0):
    """
    Vrací (nový seznam bez hodnot pod mezí, počet odstraněných).
    Vstup nemodifikuje. Vyhazuje ValueError, není-li prvek číslo.
    """
    if not isinstance(mereni, list):
        raise TypeError(f"očekávám seznam, dostal jsem {type(mereni).__name__}")

    for i, x in enumerate(mereni):
        if isinstance(x, bool) or not isinstance(x, (int, float)):
            raise ValueError(f"prvek na indexu {i} není číslo: {x!r}")

    ocistene = [x for x in mereni if x >= mez]
    return ocistene, len(mereni) - len(ocistene)


data = [12, -1, -3, 15, 20, -7, 18]
vysledek, pocet = uklid_mereni(data)
print(f"{vysledek=}, {pocet=}")     # [12, 15, 20, 18], 3
print(f"vstup beze změny: {data == [12, -1, -3, 15, 20, -7, 18]}")
print(uklid_mereni([12, 5, 20], mez=10))   # ([12, 20], 1)

### Výsledky ladění

- **Funguje:** `[12,-1,-3,15,20,-7,18]` → `([12,15,20,18], 3)`. Ověřeno ručně — tři záporné.
- **Funguje:** vstupní seznam zůstává beze změny (žádný vedlejší efekt).
- **Funguje:** `mez=10` na `[12,5,20]` → `([12,20], 1)`.
- **Hraniční:** prázdný seznam → `([], 0)`. Seznam samých záporných → `([], n)`.
- **Omezení:** `bool` odmítnut zvlášť, protože dědí z `int`.
- **Poznámka:** kdyby se opravdu vyžadovala modifikace na místě, správně je `mereni[:] = ocistene`
  — přiřazení do řezu mění původní objekt, kdežto `mereni = ocistene` by přepsalo jen lokální jméno.

### Na co se doptají

- **Proč se přeskočí prvek?** Iterátor seznamu si drží celočíselný index. Po `remove` se prvky
  posunou doleva, ale index jde dál nahoru → jeden prvek se minul.
- **Jak to jde ještě opravit?** Iterovat přes kopii (`for x in mereni[:]`), jít pozpátku
  (`for i in range(len(m)-1, -1, -1)`), nebo postavit nový seznam (nejlepší).
- **Rozdíl `mereni[:] = nove` a `mereni = nove`?** První mění objekt volajícího, druhé jen lokální jméno.

---

## Úloha 2 — Počítání četností slov

*Archetyp: KeyError u slovníku (katalog #12, #14)*

### Nalezené chyby

| # | Typ | Popis |
|---|-----|-------|
| 1 | sémantická (runtime) | `pocty[slovo] += 1` na **neexistujícím klíči** → `KeyError`. Čtení musí předcházet zápis. |
| 2 | sémantická (runtime) | `for slovo, pocet in vysledek:` iteruje přes **klíče** (řetězce) → `ValueError: too many values to unpack`. Chybí `.items()`. |

Obě spadnou za běhu — tady stačí kód spustit a Python ti řekne řádek.

### Opravené a rozšířené řešení

In [ ]:
def cetnosti(veta, min_vyskyt=1):
    """
    Vrací seznam dvojic (slovo, počet) seřazený sestupně podle počtu.
    Nerozlišuje velikost písmen. Vyhazuje ValueError pro prázdný vstup.
    """
    if not isinstance(veta, str):
        raise TypeError(f"očekávám řetězec, dostal jsem {type(veta).__name__}")
    if not veta.strip():
        raise ValueError("vstupní řetězec je prázdný")

    pocty = {}
    for slovo in veta.lower().split():
        pocty[slovo] = pocty.get(slovo, 0) + 1     # .get s výchozí hodnotou

    vybrane = {s: p for s, p in pocty.items() if p >= min_vyskyt}
    return sorted(vybrane.items(), key=lambda dvojice: dvojice[1], reverse=True)


text = "Pes kocka pes myš kocka PES"
for slovo, pocet in cetnosti(text):
    print(f"{slovo}: {pocet}")
print()
print(cetnosti(text, min_vyskyt=2))

### Výsledky ladění

- **Funguje:** `"Pes kocka pes myš kocka PES"` → `[('pes',3), ('kocka',2), ('myš',1)]`. Sedí — tři psi, dvě kočky.
- **Funguje:** `min_vyskyt=2` odfiltruje `myš`.
- **Funguje:** sjednocení velikosti písmen přes `.lower()` — `Pes`, `pes` i `PES` padnou do jednoho klíče.
- **Hraniční:** `""` a `"   "` → `ValueError`. Jedno slovo → jedna dvojice.
- **Omezení:** interpunkce se neodstraňuje, takže `"pes."` je jiné slovo než `"pes"`.
  Šlo by dořešit přes `str.strip(".,!?")`, zadání to nežádalo.
- **Poznámka:** řazení je stabilní, takže slova se stejnou četností zůstanou v pořadí prvního výskytu.

### Na co se doptají

- **Proč `.get(slovo, 0)` a ne `try/except KeyError`?** Kratší a rychlejší v běžném případě.
  Alternativy: `dict.setdefault`, `collections.defaultdict(int)`, `collections.Counter`.
- **Jaká je složitost?** Průchod slov $O(n)$, každý zápis do slovníku $O(1)$. Řazení na konci $O(k \log k)$.
- **Proč může být klíčem řetězec, ale ne seznam?** Klíč musí být hashovatelný, tedy neměnitelný.
  Seznam by po změně změnil hash a hodnota by se „ztratila".
- **Co dělá `key=lambda d: d[1]`?** Určuje, podle čeho se řadí — podle druhé složky dvojice, tj. počtu.

---

## Úloha 3 — Kopie rozvrhu

*Archetyp: mělká vs. hluboká kopie (katalog #7, #8)*

### Nalezené chyby

| # | Typ | Popis |
|---|-----|-------|
| 1 | sémantická | `kopie = rozvrh` **není kopie** — jen druhé jméno pro tentýž objekt. Jakákoli změna se projeví v obou. |
| 2 | návrhová | `import copy` je nevyužitý — nápověda, že se má použít `deepcopy`. |

**Co kód udělá:** `zal` a `rozvrh` jsou tentýž slovník, takže se do „zálohy" propíše
i nový předmět, i nový den. Kontrolní `print` vypíše dvakrát totéž.

### Opravené a rozšířené řešení

In [ ]:
import copy

def zaloha(rozvrh):
    """Vrací nezávislou hlubokou kopii rozvrhu (den -> seznam předmětů)."""
    if not isinstance(rozvrh, dict):
        raise TypeError(f"očekávám slovník, dostal jsem {type(rozvrh).__name__}")
    for den, predmety in rozvrh.items():
        if not isinstance(predmety, list):
            raise ValueError(f"hodnota u klíče {den!r} není seznam: {predmety!r}")
    return copy.deepcopy(rozvrh)


def porovnej(a, b):
    """Vrací seřazený seznam dnů, ve kterých se rozvrhy liší."""
    return sorted({d for d in a.keys() | b.keys() if a.get(d) != b.get(d)})


rozvrh = {"po": ["APR", "MAT"], "ut": ["OPS"]}
zal = zaloha(rozvrh)

rozvrh["po"].append("ZEL")
rozvrh["st"] = ["URDB"]

print("original:", rozvrh)
print("zaloha:  ", zal)
print("liší se v:", porovnej(rozvrh, zal))

In [ ]:
# Tři varianty vedle sebe — tohle si u zkoušky nakresli
puvodni = {"po": ["APR", "MAT"]}

odkaz  = puvodni                  # 1) totéž
melka  = puvodni.copy()           # 2) nový slovník, SDÍLENÉ seznamy
hlub   = copy.deepcopy(puvodni)   # 3) všechno nové

puvodni["po"].append("ZEL")       # změna VNITŘNÍHO seznamu
puvodni["ut"] = ["OPS"]           # změna VNĚJŠÍHO slovníku

print("puvodni:", puvodni)
print("odkaz:  ", odkaz,  " <- prosákly obě změny")
print("melka:  ", melka,  " <- prosákla jen vnitřní")
print("hluboka:", hlub,   " <- neprosáklo nic")

### Výsledky ladění

- **Funguje:** po `deepcopy` zůstává záloha `{'po': ['APR','MAT'], 'ut': ['OPS']}` i po změnách originálu.
- **Funguje:** `porovnej` vrací `['po', 'st']` — `po` má navíc ZEL, `st` v záloze vůbec není.
- **Klíčové pozorování:** mělká kopie ochrání **jen vnější slovník**. Přidání dne se do ní nepropíše,
  ale přidání předmětu do existujícího dne ano — protože vnitřní seznamy jsou sdílené.
- **Hraniční:** prázdný slovník projde. Slovník s neseznamovou hodnotou → `ValueError`.
- **Omezení:** `deepcopy` je pomalejší a u velkých struktur paměťově náročný. Pro plochý slovník
  řetězců by stačil `.copy()`.

### Na co se doptají

- **Kdy stačí mělká kopie?** Když jsou hodnoty **neměnitelné** (čísla, řetězce, n-tice) —
  pak není co sdílet.
- **Jak zjistím, že jde o tentýž objekt?** `a is b`, nebo `id(a) == id(b)`.
- **Co udělá `dict(rozvrh)` nebo `{**rozvrh}`?** Totéž co `.copy()` — mělkou kopii.
- **Zvládne `deepcopy` cyklický odkaz?** Ano, drží si už zkopírované objekty a nezacyklí se.

---

## Úloha 4 — Průměr každého druhého měření

*Archetyp: index vs. hodnota, off-by-one (katalog #4, #11)*

### Nalezené chyby

| # | Typ | Popis |
|---|-----|-------|
| 1 | sémantická | Testuje se **sudá hodnota** (`m % 2 == 0`), zadání chce **sudou pozici**. Chybí `enumerate`. |
| 2 | runtime | `soucet / pocet` spadne na `ZeroDivisionError`, když žádný prvek nevyhoví (např. samé liché hodnoty nebo prázdný seznam). |

**Co kód udělá:** pro `[10,3,20,7,30]` sečte 10+20+30 = 60 a vydělí 3 → `20.0`.
**Vyjde správně, ale ze špatného důvodu** — všechna čísla na sudých pozicích tu náhodou
jsou sudá. Zkus `[11, 3, 21, 7, 31]` a dostaneš `ZeroDivisionError`.

Tohle je nejzákeřnější druh chyby: **test na šťastných datech projde**. Proto vždycky
zkus vstup, kde se obě interpretace rozcházejí.

### Opravené a rozšířené řešení

In [ ]:
def prumer_kazdeho(mereni, krok=2):
    """
    Vrací (průměr prvků na indexech 0, krok, 2*krok, …, počet započtených).
    Vyhazuje ValueError pro prázdný vstup nebo nekladný krok.
    """
    if not isinstance(mereni, list):
        raise TypeError(f"očekávám seznam, dostal jsem {type(mereni).__name__}")
    if not mereni:
        raise ValueError("prázdný seznam — není z čeho počítat průměr")
    if krok < 1:
        raise ValueError(f"krok musí být kladný, dostal jsem {krok}")

    for i, x in enumerate(mereni):
        if isinstance(x, bool) or not isinstance(x, (int, float)):
            raise ValueError(f"prvek na indexu {i} není číslo: {x!r}")

    vybrane = [x for i, x in enumerate(mereni) if i % krok == 0]
    return sum(vybrane) / len(vybrane), len(vybrane)


data = [10, 3, 20, 7, 30]
print(prumer_kazdeho(data))              # (20.0, 3)
print(prumer_kazdeho(data, krok=1))      # (14.0, 5)
print(prumer_kazdeho([11, 3, 21, 7, 31]))  # (21.0, 3) — data, kde původní kód spadl

In [ ]:
# Varianta přes slicing — kratší, ale nedostaneš index
def prumer_slicingem(mereni, krok=2):
    vybrane = mereni[::krok]
    if not vybrane:
        raise ValueError("po výběru nezůstal žádný prvek")
    return sum(vybrane) / len(vybrane), len(vybrane)


# ověření, že obě varianty souhlasí
for vzorek in ([10, 3, 20, 7, 30], [1], [1, 2], [5, 5, 5, 5]):
    for k in (1, 2, 3):
        a = prumer_kazdeho(vzorek, k)
        b = prumer_slicingem(vzorek, k)
        assert a == b, f"neshoda pro {vzorek} krok {k}: {a} vs {b}"
print("obě varianty souhlasí na všech vzorcích")

### Výsledky ladění

- **Funguje:** `[10,3,20,7,30]` → `(20.0, 3)`, tedy `(10+20+30)/3`. Ověřeno ručně.
- **Funguje:** `krok=1` vezme všechno → `(14.0, 5)`, protože `(10+3+20+7+30)/5 = 14`.
- **Funguje:** `[11,3,21,7,31]` → `(21.0, 3)`. Přesně tenhle vstup původní kód shodil —
  je to důkaz, že chyba „index vs. hodnota" je opravdu opravená.
- **Funguje:** obě varianty (`enumerate` i slicing) dávají shodné výsledky, ověřeno `assert`.
- **Hraniční:** prázdný seznam → `ValueError` místo `ZeroDivisionError`. `krok=0` → `ValueError`.
  Jednoprvkový seznam → průměr je ten prvek.
- **Omezení:** průměr vrací `float` i pro celočíselný vstup, což je záměr (`/` je pravé dělení).

### Na co se doptají

- **Proč původní kód na testovacích datech prošel?** Náhoda — čísla na sudých pozicích byla shodou
  okolností sudá. Proto se testuje na datech, kde se hypotézy rozcházejí.
- **Kdy `enumerate` a kdy slicing?** Slicing je kratší, ale ztratíš index a vytvoří nový seznam.
  `enumerate` použij, když index potřebuješ v podmínce nebo ve výstupu.
- **Jaká je paměťová složitost?** `enumerate` je líný, ale comprehension stejně postaví seznam →
  $O(n/krok)$. Beze seznamu by šlo `sum(x for i,x in enumerate(m) if i%krok==0)` — generátor, $O(1)$.
- **`/` vs. `//`?** `/` vrací `float` vždy, `//` je celočíselné dělení dolů. Pro průměr chceš `/`.

---

## Úloha 5 — Sbírání položek do košíku

*Archetyp: mutable default argument (katalog #13)*

### Nalezené chyby

| # | Typ | Popis |
|---|-----|-------|
| 1 | sémantická | **Mutable default argument.** Seznam `[]` se vytvoří **jednou při definici funkce**, ne při každém volání. Všechna volání bez košíku pak sdílejí tentýž seznam. |

**Co kód udělá:** Bob dostane `['chleba', 'mléko']` — Annin chleba se mu propašuje do košíku.
Navíc `kosik_anny is kosik_boba` je `True`, takže i Annin košík se změnil.

Tohle je klasická pohovorová otázka a v katalogu chyb patří mezi ty, které
**se čtením kódu skoro nedají odhalit** — vypadá to úplně nevinně.

### Opravené a rozšířené řešení

In [ ]:
def pridej(polozka, mnozstvi=1, kosik=None):
    """
    Přidá položku do košíku (slovník položka -> počet).
    Když košík není zadán, založí NOVÝ. Vrací košík.
    """
    if not isinstance(polozka, str) or not polozka.strip():
        raise ValueError(f"název položky musí být neprázdný řetězec, dostal jsem {polozka!r}")
    if not isinstance(mnozstvi, int) or isinstance(mnozstvi, bool) or mnozstvi < 1:
        raise ValueError(f"množství musí být kladné celé číslo, dostal jsem {mnozstvi!r}")

    if kosik is None:          # ← klíčová oprava
        kosik = {}

    kosik[polozka] = kosik.get(polozka, 0) + mnozstvi
    return kosik


def odeber(kosik, polozka, mnozstvi=1):
    """Sníží počet kusů; při nule klíč odstraní. Neznámá položka -> KeyError."""
    if polozka not in kosik:
        raise KeyError(f"položka {polozka!r} v košíku není")
    kosik[polozka] -= mnozstvi
    if kosik[polozka] <= 0:
        del kosik[polozka]
    return kosik


def celkem(kosik):
    """Součet všech kusů v košíku."""
    return sum(kosik.values())

In [ ]:
kosik_anny = pridej("chleba")
kosik_boba = pridej("mléko")
print("Anna:", kosik_anny)      # {'chleba': 1}
print("Bob: ", kosik_boba)      # {'mléko': 1}   ← už se nemíchají
print("sdílejí objekt?", kosik_anny is kosik_boba)   # False

pridej("chleba", 2, kosik_anny)
pridej("máslo", mnozstvi=3, kosik=kosik_anny)
print("Anna:", kosik_anny, "celkem", celkem(kosik_anny))   # {'chleba': 3, 'máslo': 3} celkem 6

odeber(kosik_anny, "chleba", 3)
print("po odebrání:", kosik_anny)    # {'máslo': 3}

try:
    odeber(kosik_anny, "rohlík")
except KeyError as e:
    print("KeyError:", e)

### Výsledky ladění

- **Funguje:** dvě samostatná volání bez košíku vrací **nezávislé** slovníky —
  ověřeno `is`, vrací `False`. Tím je původní chyba prokazatelně opravená.
- **Funguje:** opakované přidání téže položky sčítá množství (`chleba` 1+2 = 3).
- **Funguje:** `odeber` při dosažení nuly klíč smaže, ne nechá `0`.
- **Funguje:** `celkem` sečte hodnoty přes `sum(kosik.values())` → 6 pro 3+3.
- **Hraniční:** prázdný název, `mnozstvi=0` i záporné → `ValueError`. Odebrání víc kusů,
  než je v košíku, položku smaže (nejde do záporu).
- **Omezení:** `odeber` neřeší, když se odebere víc, než je k dispozici — bere to jako „smaž".
  Kdyby zadání chtělo jinak, přidal bych kontrolu `mnozstvi > kosik[polozka]`.
- **Poznámka:** `isinstance(mnozstvi, bool)` je tam schválně — `True` by jinak prošlo jako `1`.

### Na co se doptají

- **Proč se výchozí hodnota vytvoří jen jednou?** `def` je příkaz, který se vykoná jednou
  při definici. Výchozí hodnoty se tehdy vyhodnotí a uloží do `funkce.__defaults__`.
- **Které typy jsou v defaultu nebezpečné?** Všechny měnitelné: `list`, `dict`, `set`.
  Neměnitelné (`int`, `str`, `tuple`, `None`) jsou v pohodě.
- **Jak to ověřím?** `print(pridej.__defaults__)` — u chybné verze uvidíš seznam, který roste.
- **Proč `is None` a ne `== None`?** `is` porovnává identitu a nedá se přebít vlastním `__eq__`.
- **Dá se to využít schválně?** Ano, jako primitivní cache mezi voláními — ale je to matoucí,
  lepší je `functools.lru_cache`.

---

## Úloha 6 — Seřazení a normalizace jmen

*Archetyp: metody vracející None (katalog #9, #10)*

### Nalezené chyby

| # | Typ | Popis |
|---|-----|-------|
| 1 | sémantická | `ocistena = ocistena.sort()` — `sort()` řadí **na místě a vrací `None`**. Přiřazením si přepíšeš seznam na `None`. |
| 2 | důsledek | Funkce vrací `None` místo seznamu. |

**Co kód udělá:** vypíše `None`. Kdyby tam bylo jen `ocistena.sort()` bez přiřazení,
fungovalo by to — chyba je v tom přiřazení.

**Pravidlo:** metody, které mění na místě (`sort`, `reverse`, `append`, `extend`, `remove`,
`insert`, `clear`) vrací `None`. Vestavěné funkce, které vrací nový objekt, jsou
`sorted`, `reversed`, `list`.

### Opravené a rozšířené řešení

In [ ]:
def uprav_jmena(jmena, sestupne=False, min_delka=1):
    """
    Vrací NOVÝ seřazený seznam normalizovaných jmen bez duplicit.
    Vyhazuje ValueError pro neřetězcový nebo prázdný prvek.
    """
    if not isinstance(jmena, list):
        raise TypeError(f"očekávám seznam, dostal jsem {type(jmena).__name__}")

    ocistena = []
    for i, j in enumerate(jmena):
        if not isinstance(j, str):
            raise ValueError(f"prvek na indexu {i} není řetězec: {j!r}")
        normalizovane = j.strip().capitalize()
        if not normalizovane:
            raise ValueError(f"prvek na indexu {i} je po očištění prázdný")
        if len(normalizovane) >= min_delka:
            ocistena.append(normalizovane)

    return sorted(set(ocistena), reverse=sestupne)    # sorted VRACÍ nový seznam


vstup = ["  novák ", "DVOŘÁK", "  Černý", "NOVÁK", "Ab"]
print(uprav_jmena(vstup))                    # ['Ab', 'Dvořák', 'Novák', 'Černý']
print(uprav_jmena(vstup, sestupne=True))
print(uprav_jmena(vstup, min_delka=3))       # 'Ab' vypadne
print("vstup beze změny:", vstup[0] == "  novák ")

### Výsledky ladění

- **Funguje:** `['  novák ', 'DVOŘÁK', '  Černý', 'NOVÁK', 'Ab']` → `['Ab', 'Dvořák', 'Novák', 'Černý']`.
  Duplicita `novák`/`NOVÁK` se po normalizaci sloučila.
- **Funguje:** `sestupne=True` obrátí pořadí, `min_delka=3` vyhodí `Ab`.
- **Funguje:** vstupní seznam zůstává beze změny — pracuje se s novým seznamem.
- **Hraniční:** prázdný seznam → `[]`. Prvek `"   "` → `ValueError` (po očištění prázdný).
- **Omezení (důležité!):** řazení je podle **Unicode kódů**, ne podle české abecedy —
  proto `Černý` skončí **až za** `Novák`. Správné české řazení by chtělo `locale.strxfrm`
  nebo knihovnu `PyICU`. **Tohle u obhajoby zmiň sám**, je to viditelná vada výstupu.
- **Omezení:** `capitalize()` zvládne jen jednoslovná jména — `"jan novák"` udělá `"Jan novák"`.
  Pro víceslovná je `str.title()`, ale to zase rozbije `"O'Brien"`.

### Na co se doptají

- **Které metody vrací `None`?** Ty, co mění na místě: `sort`, `reverse`, `append`, `extend`,
  `insert`, `remove`, `clear`. Protipóly vracející nový objekt: `sorted`, `reversed`, `list`, `copy`.
- **Proč to tak Python dělá?** Konvence: funkce buď mění objekt, nebo vrací nový — ne obojí.
  Vrácení `None` je signál „změnil jsem to na místě".
- **Proč `set()` a pak `sorted()`, ne naopak?** `set` pořadí nezachovává, takže seřadit se musí až potom.
- **Jak odstranit duplicity a zachovat pořadí prvního výskytu?** `list(dict.fromkeys(seznam))` —
  slovník si od Pythonu 3.7 pamatuje pořadí vložení.
- **Jaká je složitost?** Normalizace $O(n)$, `set` $O(n)$, řazení $O(n \log n)$ → celkem $O(n \log n)$.

---